In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="llama3.2:latest")
print("Model is ready")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
import json

def ollama_eval(model_name, dataset):
    """
    Runs evaluation for a given Ollama model against a dataset.
    
    Args:
        model_name (str): Name of the Ollama model (e.g., 'llama2', 'mistral').
        dataset (list): List of dicts with 'prompt' and 'expected' keys.
    
    Returns:
        dict: Evaluation results including accuracy and detailed outputs.
    """
    results = []
    correct_count = 0

    for idx, item in enumerate(dataset, start=1):
        prompt = item["prompt"]
        expected = item["expected"]

        user_chat_prompt_template = ChatPromptTemplate([
        ("user", prompt)
        ])

        user_prompt = user_chat_prompt_template.invoke({})
        output_text = model.invoke(user_prompt).content

        print(f'output: {output_text}')

        # Compare with expected
        is_correct = output_text.lower() == expected.lower()
        if is_correct:
            correct_count += 1

        results.append({
            "id": idx,
            "prompt": prompt,
            "expected": expected,
            "output": output_text,
            "correct": is_correct
        })

        accuracy = correct_count / len(dataset) if dataset else 0.0
        return {
        "model": model_name,
        "accuracy": accuracy,
        "results": results
        }

if __name__ == "__main__":
    # Example dataset for evaluation
    sample_dataset = [
        {"prompt": "What is 2+2?", "expected": "2 + 2 = 4"},
        {"prompt": "Capital of France?", "expected": "Paris"},
        {"prompt": "Translate 'Hello' to Spanish.", "expected": "Hola"}
    ]

    model_to_test = "llama3.2:latest"  # Change to your installed Ollama model
    eval_report = ollama_eval(model_to_test, sample_dataset)

    # Pretty print results
    print(json.dumps(eval_report, indent=2))


In [ ]:
import requests
import json


# Ollama server URL (default local install)
OLLAMA_API_URL = "http://localhost:11434/api/generate"
#OLLAMA_API_URL = "http://ollama.com/api/generate"


def ollama_eval(model_name, dataset):
    """
    Runs evaluation for a given Ollama model against a dataset.
    
    Args:
        model_name (str): Name of the Ollama model (e.g., 'llama2', 'mistral').
        dataset (list): List of dicts with 'prompt' and 'expected' keys.
    
    Returns:
        dict: Evaluation results including accuracy and detailed outputs.
    """
    results = []
    correct_count = 0

    for idx, item in enumerate(dataset, start=1):
        prompt = item["prompt"]
        expected = item["expected"]

        try:
            # Send request to Ollama
            response = requests.post(
                OLLAMA_API_URL,
                json={"model": model_name, "prompt": prompt},
                timeout=30
            )

            response.raise_for_status()
            output_text = response.json().get("response", "").strip()
            print(f'output: {output_text}')

            # Compare with expected
            is_correct = output_text.lower() == expected.lower()
            if is_correct:
                correct_count += 1

            results.append({
                "id": idx,
                "prompt": prompt,
                "expected": expected,
                "output": output_text,
                "correct": is_correct
            })

        except requests.exceptions.RequestException as e:
            results.append({
                "id": idx,
                "prompt": prompt,
                "expected": expected,
                "output": None,
                "correct": False,
                "error": str(e)
            })

    accuracy = correct_count / len(dataset) if dataset else 0.0
    return {
        "model": model_name,
        "accuracy": accuracy,
        "results": results
    }

if __name__ == "__main__":
    # Example dataset for evaluation
    sample_dataset = [
        {"prompt": "What is 2+2?", "expected": "4"},
        {"prompt": "Capital of France?", "expected": "Paris"},
        {"prompt": "Translate 'Hello' to Spanish.", "expected": "Hola"}
    ]

    model_to_test = "llama3.2:latest"  # Change to your installed Ollama model
    eval_report = ollama_eval(model_to_test, sample_dataset)

    # Pretty print results
    print(json.dumps(eval_report, indent=2))
